# Day 2 | Lab 2.2: Foundational Prompt Patterns

**Duration:** ~1.5 hours

**Scenario.** Retail-banking + investment products — preserved from the GM source notebook (banking already present). This lab introduces four production-grade prompt patterns: **persona**, **N-shot prompting**, **directional stimulus**, and **output template**.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Apply a **persona** to set tone, expertise level, and audience.
2. Use **N-shot examples** to lock in output format and decision boundaries.
3. Add **directional stimulus** to focus the model on specific aspects of input.
4. Enforce a **structured output template** (JSON, table, fixed-section) that downstream code can parse.
5. Scale few-shot examples for production using `FewShotChatMessagePromptTemplate` instead of inlining examples in a string.

*Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'langchain>=1.0' 'langchain-core>=1.0' 'langchain-openai>=1.0' python-dotenv


In [1]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv("..\\.env")
except ImportError:
    pass

for key in ['OPENAI_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


OPENAI_API_KEY: ✅ Loaded


In [2]:
from IPython.display import display, Image, Markdown

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# # Load environment variables from a .env file
# load_dotenv()

# Initialize the model
llm = ChatOpenAI(model="gpt-5-mini")

# Helper function to execute prompts and print results
def execute_prompt(prompt_template, title, **kwargs):
    """
    Executes a given prompt template and prints the output neatly.
    """
    print(f"--- {title} ---")

    prompt = ChatPromptTemplate.from_template(prompt_template)
    chain = prompt | llm | StrOutputParser()

    print("PROMPT:")
    print(prompt.format(**kwargs))

    response = chain.invoke(kwargs)

    print("\n" + "="*20 + " RESPONSE " + "="*20 + "\n")
    display(Markdown(response))
    print("\n" + "-"*50 + "\n")

-----

### **1. The Persona Pattern**

**Background:** The **Persona Pattern** involves instructing the LLM to "act as" a specific role or character. This is one of the most effective ways to control the tone, style, and expertise level of the response. Assigning a persona primes the model to generate content that aligns with the assumed role's knowledge and communication style. 🎭

**Scenario:** A bank needs to draft a marketing email to a high-net-worth individual (HNI) about a new, exclusive investment fund. The tone must be professional, confident, and sophisticated.

In [4]:
fund_details = """
- Fund Name: "Apex Global Growth Fund"
- Strategy: Invests in high-growth international technology and healthcare equities.
- Target Audience: Sophisticated investors with a high risk appetite.
- Minimum Investment: ₹1 Crore.
"""

In [5]:
# The output without StrOutputParser()
naive_persona_prompt = "Summarize Generative AI in 3 bullet points"
prompt = ChatPromptTemplate.from_template(naive_persona_prompt)
chain = prompt | llm
response = chain.invoke({})
response

AIMessage(content='- What it is: AI systems (e.g., large language and image models) that learn patterns from large datasets and generate novel content—text, images, audio, code—rather than only classifying or extracting information.\n- How it works: neural networks (commonly Transformer-based for text and diffusion/GANs for images) are trained to predict next tokens or reverse noise processes, enabling them to produce coherent, diverse outputs conditioned on prompts.\n- Why it matters (and risks): enables automation, creativity, personalized content and faster prototyping, but raises issues like hallucination, bias, copyright, safety and misuse—so human oversight, evaluation and guardrails are essential.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 337, 'prompt_tokens': 17, 'total_tokens': 354, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'pr

In [6]:
# The output without StrOutputParser()
naive_persona_prompt = "Summarize Generative AI in 3 bullet points"
prompt = ChatPromptTemplate.from_template(naive_persona_prompt)
chain = prompt | llm | StrOutputParser()
response = chain.invoke({})
response

'- What it is: Machine-learning models (e.g., transformers, diffusion models, GANs) that learn patterns from large datasets and generate novel content—text, images, audio, code, or video—rather than just classifying or predicting.  \n- What it does: Enables rapid content creation, prototyping, personalization, data augmentation and automation across industries (creative work, software development, R&D, marketing, education, simulation).  \n- Key risks and limits: Prone to hallucinations, bias, copyright/privacy issues and misuse (deepfakes, misinformation); requires human oversight, verification, robust evaluation and governance to deploy responsibly.'

#### **Naive Prompt 👎**

Without a persona, the model's tone is generic and lacks the authority required for the target audience.

In [7]:
naive_persona_prompt = "Write an email about a new investment fund with these details:\n\n{details}"

execute_prompt(
    naive_persona_prompt,
    "1.1 Naive Email Draft",
    details=fund_details
)

--- 1.1 Naive Email Draft ---
PROMPT:
Human: Write an email about a new investment fund with these details:


- Fund Name: "Apex Global Growth Fund"
- Strategy: Invests in high-growth international technology and healthcare equities.
- Target Audience: Sophisticated investors with a high risk appetite.
- Minimum Investment: ₹1 Crore.


==================== RESPONSE ====================



Subject: Introducing Apex Global Growth Fund — High-Growth International Tech & Healthcare Equities

Dear [Name],

I’m pleased to introduce the Apex Global Growth Fund, a new investment opportunity designed for sophisticated investors seeking high-growth exposure to international technology and healthcare equities.

Fund overview
- Name: Apex Global Growth Fund  
- Strategy: Active investment in high-growth international technology and healthcare companies. The Fund focuses on secular innovation themes across software, artificial intelligence, digital infrastructure, biotechnology and specialty therapeutics, with a portfolio constructed to capture long-term structural growth outside India.  
- Target investors: Sophisticated investors with a high risk appetite who are comfortable with equity volatility and concentrated, growth-oriented portfolios.  
- Minimum investment: ₹1 Crore

Why consider this opportunity
- The Fund seeks to identify differentiated, high-growth companies poised to benefit from transformative technologies and medical advances.  
- International diversification across developed and select emerging markets to capture innovation leaders outside the domestic market.  
- Designed for investors with a long-term investment horizon who can tolerate higher short-term volatility in pursuit of above-average growth.

Next steps
If you would like the Confidential Private Placement Memorandum (PPM), a fund presentation, or to schedule a call to discuss suitability and subscription mechanics, please reply to this email or contact me at [phone number/email]. We can provide full fund documentation and walk through risk factors, fees, governance and the subscription process.

Important risk notice
The Fund is intended for sophisticated investors only. Investments in equities involve market risks and the value of your capital may fluctuate; principal is not guaranteed. Past performance is not indicative of future results. Please review the offering documents and consult your financial, tax and legal advisors before investing.

Thank you for your interest. I look forward to speaking with you.

Sincerely,  
[Your Name]  
[Title]  
[Firm Name]  
[Phone] | [Email]


--------------------------------------------------



*Critique:* This is dry, unprofessional, and fails to convey exclusivity or expertise. It would not appeal to a sophisticated investor.

#### **Persona Pattern Prompt 👍**

By assigning the persona of a **"Senior Wealth Manager,"** we instruct the model to adopt a suitable tone and frame the message appropriately.

In [8]:
persona_prompt = """
Act as a Senior Wealth Manager at a prestigious private bank.
Your task is to draft a personalized email to a long-standing, high-net-worth client.

The email should be sophisticated, concise, and compelling.
Introduce the new fund and suggest a brief meeting to discuss how it aligns with their portfolio goals.

Use the following fund details:
{details}
"""

execute_prompt(
    persona_prompt,
    "1.2 Persona-Driven Email Draft",
    details=fund_details
)

--- 1.2 Persona-Driven Email Draft ---
PROMPT:
Human: 
Act as a Senior Wealth Manager at a prestigious private bank.
Your task is to draft a personalized email to a long-standing, high-net-worth client.

The email should be sophisticated, concise, and compelling.
Introduce the new fund and suggest a brief meeting to discuss how it aligns with their portfolio goals.

Use the following fund details:

- Fund Name: "Apex Global Growth Fund"
- Strategy: Invests in high-growth international technology and healthcare equities.
- Target Audience: Sophisticated investors with a high risk appetite.
- Minimum Investment: ₹1 Crore.



==================== RESPONSE ====================



Subject: Introducing Apex Global Growth Fund — a high‑conviction opportunity in global tech & healthcare

Dear [Client Name],

I hope you are well. I wanted to share a new, invitation‑only opportunity that may suit your objective of long‑term capital appreciation: the Apex Global Growth Fund.

At a glance
- Strategy: concentrated, high‑conviction investments in high‑growth international technology and healthcare equities
- Profile: designed for sophisticated investors with a high risk appetite seeking accelerated growth and thematic global exposure
- Minimum investment: ₹1 Crore

The fund is actively managed by a team focused on secular innovation trends across software, AI, biotechnology and advanced therapeutics — areas we believe can meaningfully complement a growth‑oriented sleeve of your portfolio while providing differentiated international exposure.

If convenient, I suggest a brief 20–30 minute call to review how this fits with your objectives, time horizon and risk profile, and to share the detailed fund memorandum and performance expectations. I am available next week on Tuesday or Thursday afternoon, or please propose a time that suits you.

I look forward to discussing this with you. 

Warm regards,

[Your Name]  
Senior Wealth Manager  
[Bank Name] — Private Banking  
Direct: [phone] | Email: [email]


--------------------------------------------------



*Critique:* Excellent. The tone is professional, the language is sophisticated ("investment vehicle," "forward-thinking portfolio strategy"), and the call-to-action is respectful of the client's time.

-----

### **2. The N-Shot Prompting Pattern**

**Background:** This pattern involves providing the model with examples ("shots") to guide its output.

  * **Zero-Shot:** You provide no examples, just the instruction. This works for simple, well-defined tasks.
  * **Few-Shot:** You provide a few examples of input and desired output. This is powerful for teaching the model a specific format, structure, or nuanced classification.

#### **2.1 Zero-Shot Use Case: Customer Request Classification**

**Scenario:** A banking chatbot receives an incoming customer message. It needs to classify the message into a predefined category for routing to the correct department.

In [9]:
customer_message = "Hi, my credit card was charged twice for a purchase I made at BigBasket yesterday. Can you please help me fix this?"

**Improved Zero-Shot Prompt 👍**
We don't need examples here because the task is a straightforward classification. We just need to give clear instructions and the list of valid categories.

In [10]:
zeroshot_prompt = """
You are a classification bot.
Classify the following customer message into one of these categories:
- Account Inquiry
- Card Dispute
- Loan Application
- Technical Support

Return only the category name.

Message: "{message}"
"""

execute_prompt(
    zeroshot_prompt,
    "2.1 Zero-Shot Classification",
    message=customer_message
)

--- 2.1 Zero-Shot Classification ---
PROMPT:
Human: 
You are a classification bot.
Classify the following customer message into one of these categories:
- Account Inquiry
- Card Dispute
- Loan Application
- Technical Support

Return only the category name.

Message: "Hi, my credit card was charged twice for a purchase I made at BigBasket yesterday. Can you please help me fix this?"


==================== RESPONSE ====================



Card Dispute


--------------------------------------------------



*Critique:* This is a perfect use of zero-shot prompting. The task is clear, and the model can easily perform it without needing examples.

#### **2.2 Few-Shot Use Case: Structuring Trade Data**

**Scenario:** A system needs to parse unstructured text from a trader's chat message and convert it into a structured format for the order management system. The format must be exact.

In [11]:
trader_chat = "we need to buy 5000 shares of RELIANCE at market price asap"

#### **Naive Prompt 👎**

A naive prompt will result in an unpredictable structure. The keys and values might change with every run.

In [12]:
naive_fewshot_prompt = "Extract the details from this trade message: {message}"

execute_prompt(
    naive_fewshot_prompt,
    "2.2 Naive Trade Extraction",
    message=trader_chat
)

--- 2.2 Naive Trade Extraction ---
PROMPT:
Human: Extract the details from this trade message: we need to buy 5000 shares of RELIANCE at market price asap

==================== RESPONSE ====================



Extracted order details:
- Action: Buy
- Quantity: 5,000 shares
- Instrument/ticker: RELIANCE
- Price type: Market
- Execution urgency: ASAP (execute immediately)

Assumptions / missing items you may want to confirm:
- Exchange/venue (which market for RELIANCE)
- Account / client ID to route the order
- Time-in-force / partial-fill preference (ASAP could be IOC/FOK or standard Market Day; confirm if partial fills are allowed)
- Currency/settlement instructions (if needed)
- Any additional order instructions (e.g., minimum fill, routing preference)

If you want, I can format this as an order ticket (with suggested TIF and partial-fill setting) ready to send to the broker.


--------------------------------------------------



*Critique:* This is just a string, not a machine-readable format. It's also missing a key for the price.

#### **Few-Shot Pattern Prompt 👍**

We provide a clear example to teach the model the **exact output format** we need.

In [13]:
fewshot_prompt = """
You are an AI that converts unstructured trader chat messages into a structured string.

Follow the format of the example precisely.
---
**Example:**
Message: "sell 200 INFY limit 1500"
Output: "ACTION:SELL|TICKER:INFY|QUANTITY:200|PRICE:1500.00"
---

**New Message:**
Message: "{message}"
Output:
"""

execute_prompt(
    fewshot_prompt,
    "2.3 Few-Shot Trade Extraction",
    message=trader_chat
)

--- 2.3 Few-Shot Trade Extraction ---
PROMPT:
Human: 
You are an AI that converts unstructured trader chat messages into a structured string.

Follow the format of the example precisely.
---
**Example:**
Message: "sell 200 INFY limit 1500"
Output: "ACTION:SELL|TICKER:INFY|QUANTITY:200|PRICE:1500.00"
---

**New Message:**
Message: "we need to buy 5000 shares of RELIANCE at market price asap"
Output:


==================== RESPONSE ====================



"ACTION:BUY|TICKER:RELIANCE|QUANTITY:5000|PRICE:MKT"


--------------------------------------------------



*Critique:* Perfect. The few-shot example forced the model to produce an output that is structured, consistent, and ready for a downstream system to parse.

-----


---
## Few-Shot at Scale with `FewShotChatMessagePromptTemplate`

The few-shot prompt above hard-codes its examples inside a single string. That's fine for 2-3 examples, but breaks down for production:
- You can't easily add/remove examples without re-editing the string
- You can't conditionally select examples per input (e.g., closest-match retrieval)
- You can't separately version-control your example library

LangChain's `FewShotChatMessagePromptTemplate` separates **examples** (data) from **prompt structure** (template). This is the production-scalable pattern.


In [14]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate,
)

# 1. Examples live as data — easy to grow, swap, or load from JSON / DB.
examples = [
    {
        'input': 'we need to buy 5000 shares of RELIANCE at market price asap',
        'output': '''```json
{
  "action": "BUY",
  "ticker": "RELIANCE",
  "quantity": 5000,
  "order_type": "MARKET",
  "urgency": "HIGH"
}
```'''
    },
    {
        'input': 'sell 200 INFY at limit 1850',
        'output': '''```json
{
  "action": "SELL",
  "ticker": "INFY",
  "quantity": 200,
  "order_type": "LIMIT",
  "limit_price": 1850,
  "urgency": "NORMAL"
}
```'''
    },
]

# 2. Each example becomes a (Human, AI) message pair.
example_prompt = ChatPromptTemplate.from_messages([
    ('human', '{input}'),
    ('ai', '{output}'),
])

# 3. Wrap into a few-shot template — handles iteration over examples for you.
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# 4. Assemble the final prompt — system + few-shot block + user input.
final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You convert short trader-chat strings to structured trade orders in JSON.'),
    few_shot_prompt,
    ('human', '{input}'),
])

print(final_prompt.format(input='buy 1000 TCS at market'))


System: You convert short trader-chat strings to structured trade orders in JSON.
Human: we need to buy 5000 shares of RELIANCE at market price asap
AI: ```json
{
  "action": "BUY",
  "ticker": "RELIANCE",
  "quantity": 5000,
  "order_type": "MARKET",
  "urgency": "HIGH"
}
```
Human: sell 200 INFY at limit 1850
AI: ```json
{
  "action": "SELL",
  "ticker": "INFY",
  "quantity": 200,
  "order_type": "LIMIT",
  "limit_price": 1850,
  "urgency": "NORMAL"
}
```
Human: buy 1000 TCS at market


In [15]:
# Run it on a new trader chat — model should follow the JSON schema established by the examples.
chain = final_prompt | llm | StrOutputParser()
result = chain.invoke({'input': 'sell 500 HDFC bank at 1620 limit, kind of urgent'})
print(result)


```json
{
  "action": "SELL",
  "ticker": "HDFCBANK",
  "quantity": 500,
  "order_type": "LIMIT",
  "limit_price": 1620,
  "urgency": "HIGH"
}
```


### 📝 Why this matters in production
- **Examples = data.** Store them in JSON / a database / Langfuse / LangSmith Hub and load at runtime.
- **Dynamic example selection.** Pair `FewShotChatMessagePromptTemplate` with a `SemanticSimilarityExampleSelector` to pick the *k* closest examples to the current input — significantly better than fixed examples for diverse inputs.
- **A/B testing.** Two example libraries → two chains → run both, compare quality.
- **Decoupling.** Prompt structure changes (system message, output format) don't require touching examples, and vice-versa.


---
### **3. The Directional Stimulus Pattern**

**Background:** This pattern guides the LLM to focus on specific aspects of a large piece of text. Instead of a generic summary, you provide "hints" or "directions" to tailor the output to your specific needs. 🎯

**Scenario:** A busy loan officer needs to quickly understand the key risks in a lengthy business credit report. They don't need a full summary, only the negative points.

In [20]:
credit_report_excerpt = """
Business 'Future Gadgets Ltd.' has shown consistent revenue growth of 15% YoY.
Their cash flow is positive, and they hold significant assets.
However, their debt-to-equity ratio is high at 2.5, and the report notes two recent late payments to major suppliers.
The market outlook for their sector is stable, but dependent on volatile international supply chains.
"""

#### **Naive Prompt 👎**

A generic summary will include positive points, forcing the officer to read through extra information.

In [21]:
naive_directional_prompt = "Summarize this credit report excerpt:\n\n{report}"

execute_prompt(
    naive_directional_prompt,
    "3.1 Naive Report Summary",
    report=credit_report_excerpt
)

--- 3.1 Naive Report Summary ---
PROMPT:
Human: Summarize this credit report excerpt:


Business 'Future Gadgets Ltd.' has shown consistent revenue growth of 15% YoY. 
Their cash flow is positive, and they hold significant assets. 
However, their debt-to-equity ratio is high at 2.5, and the report notes two recent late payments to major suppliers. 
The market outlook for their sector is stable, but dependent on volatile international supply chains.


==================== RESPONSE ====================



Summary — Future Gadgets Ltd.

- Positives:
  - Consistent revenue growth ~15% year-over-year.
  - Positive cash flow.
  - Holds significant assets.

- Risks / negatives:
  - High leverage: debt-to-equity ratio = 2.5.
  - Two recent late payments to major suppliers (indicative of potential liquidity or operational strain).

- External outlook:
  - Sector outlook is stable but reliant on volatile international supply chains (exposure to external disruption).

Overall takeaway: The company shows strong growth and asset backing, but elevated leverage and recent supplier payment issues raise credit risk. Recommend monitoring liquidity, supplier relationships and supply-chain exposure before extending further credit.


--------------------------------------------------



*Critique:* The summary is accurate but mixes positive and negative points.

#### **Directional Stimulus Prompt 👍**

We give the model a clear "hint" to focus *only* on the risk factors.

In [22]:
directional_prompt = """
Summarize the following credit report excerpt in 2-3 bullet points.

Hint: Focus exclusively on the negative points and potential risk factors mentioned in the report.

Report:
{report}
"""

execute_prompt(
    directional_prompt,
    "3.2 Directional Risk Summary",
    report=credit_report_excerpt
)

--- 3.2 Directional Risk Summary ---
PROMPT:
Human: 
Summarize the following credit report excerpt in 2-3 bullet points.

Hint: Focus exclusively on the negative points and potential risk factors mentioned in the report.

Report:

Business 'Future Gadgets Ltd.' has shown consistent revenue growth of 15% YoY. 
Their cash flow is positive, and they hold significant assets. 
However, their debt-to-equity ratio is high at 2.5, and the report notes two recent late payments to major suppliers. 
The market outlook for their sector is stable, but dependent on volatile international supply chains.



==================== RESPONSE ====================



- High leverage: debt-to-equity ratio of 2.5 indicates elevated financial risk and reduced buffer for downturns.  
- Payment performance concerns: two recent late payments to major suppliers, signaling cash management or liquidity strains and potential supplier/creditor relationship risk.  
- Supply-chain exposure: business is dependent on volatile international supply chains, creating operational and market risk despite a stable sector outlook.


--------------------------------------------------



*Critique:* This is much more effective. It provides the loan officer with a concise, focused list of the exact information they need, saving them time and improving efficiency.

-----

### **4. The Template & Meta Language Pattern**

**Background:** This powerful pattern involves defining a strict structure or a custom "language" for the model's output. By providing a template, you force the LLM to populate its response within your predefined format, making the output predictable and machine-readable. This is essential for JSON, XML, or other structured data needs. 📝

**Scenario:** A bank needs to generate a standardized risk assessment for a credit card application in a JSON format that can be logged directly into their system.

In [23]:
applicant_data = """
- Name: Rahul Verma
- Age: 28
- Annual Income: ₹8,00,000
- CIBIL Score: 760
- Existing Debt: ₹1,50,000 (Personal Loan)
"""

#### **Naive Prompt 👎**

Without a strict template, the model will generate JSON, but the structure, keys, and value types will be inconsistent.

In [24]:
naive_template_prompt = "Create a JSON risk summary for this credit card applicant:\n\n{data}"

execute_prompt(
    naive_template_prompt,
    "4.1 Naive JSON Generation",
    data=applicant_data
)

--- 4.1 Naive JSON Generation ---
PROMPT:
Human: Create a JSON risk summary for this credit card applicant:


- Name: Rahul Verma
- Age: 28
- Annual Income: ₹8,00,000
- CIBIL Score: 760
- Existing Debt: ₹1,50,000 (Personal Loan)


==================== RESPONSE ====================



{
  "applicant": {
    "name": "Rahul Verma",
    "age": 28,
    "annual_income_inr": 800000,
    "cibil_score": 760,
    "existing_debt_inr": 150000,
    "debt_to_income_ratio": 0.1875
  },
  "risk_assessment": {
    "risk_score_0_100": 25,
    "risk_level": "Low",
    "probability_of_default_1yr_percent": 0.9,
    "confidence": "High",
    "key_risk_drivers": [
      "Existing personal loan increases outstanding obligations",
      "Moderate annual income relative to debt burden",
      "Strong CIBIL score (760) reduces credit risk"
    ],
    "mitigating_factors": [
      "Low debt-to-income ratio (~18.8%)",
      "Young borrower with stable income (assessed)",
      "Good repayment history implied by CIBIL"
    ]
  },
  "credit_recommendation": {
    "eligibility_recommendation": "Approve (subject to standard underwriting and ID/KYC checks)",
    "recommended_credit_limit_inr": 75000,
    "recommended_credit_limit_range_inr": {
      "min": 50000,
      "max": 100000
    },
    "recommended_pricing_notes": "Standard retail APR; consider conservative limit tier given existing personal loan",
    "monitoring_recommendations": [
      "Verify current employment and income documentation",
      "Monitor credit bureau for new delinquencies or rising utilization",
      "Encourage timely repayment of existing personal loan to improve capacity"
    ]
  },
  "metadata": {
    "assessment_date": "2026-03-02",
    "model_version": "CC_Risk_v1.2"
  }
}


--------------------------------------------------



*Critique:* The keys (`applicant`, `income`, `risk`) are arbitrary, and the `risk` value is a subjective string, not a standardized category.

#### **Template Pattern Prompt 👍**

We define a "meta language" by specifying the exact keys and the allowed values for certain fields, ensuring a perfectly structured output.

In [25]:
template_prompt = """
Act as a risk assessment bot. Generate a JSON summary for the following credit card applicant.

The output must be a valid JSON object that strictly follows this template:
- "applicantId": string (use applicant's name)
- "cibilScore": integer
- "incomeToDebtRatio": float (calculate as Annual Income / Existing Debt)
- "riskLevel": string (must be one of the following: "Low", "Medium", "High")
- "isApproved": boolean (approve if score > 750 and incomeToDebtRatio > 5)

Applicant Data:
{data}
"""

execute_prompt(
    template_prompt,
    "4.2 Template-Driven JSON Generation",
    data=applicant_data
)

--- 4.2 Template-Driven JSON Generation ---
PROMPT:
Human: 
Act as a risk assessment bot. Generate a JSON summary for the following credit card applicant.

The output must be a valid JSON object that strictly follows this template:
- "applicantId": string (use applicant's name)
- "cibilScore": integer
- "incomeToDebtRatio": float (calculate as Annual Income / Existing Debt)
- "riskLevel": string (must be one of the following: "Low", "Medium", "High")
- "isApproved": boolean (approve if score > 750 and incomeToDebtRatio > 5)

Applicant Data:

- Name: Rahul Verma
- Age: 28
- Annual Income: ₹8,00,000
- CIBIL Score: 760
- Existing Debt: ₹1,50,000 (Personal Loan)



==================== RESPONSE ====================



{
  "applicantId": "Rahul Verma",
  "cibilScore": 760,
  "incomeToDebtRatio": 5.333333333333333,
  "riskLevel": "Low",
  "isApproved": true
}


--------------------------------------------------



*Critique:* This is perfect. The JSON is clean, follows the exact schema, performs a calculation as instructed, and uses a standardized value for `riskLevel`. This output can be reliably processed by another program.

---
## 8. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Persona pattern** | Sets tone, expertise, audience — must be in the system prompt |
| **N-shot pattern** | Lock format and decision boundaries the model can't infer from instructions |
| **Directional stimulus** | Direct attention to specific aspects of input — e.g. 'focus on the negative factors' |
| **Template pattern** | Enforce schema explicitly so downstream code can parse without surprises |
| **`FewShotChatMessagePromptTemplate`** | The production scalable form — examples as data, prompt as template |

**Next Lab:** Lab 2.3 — Advanced Prompting (CoT, ToT, CoVe) 🧠


## 9. Stretch Exercise (Optional)

1. Add a 5th example to the trader-chat few-shot library that demonstrates an *invalid* input (e.g., missing ticker). Test that the model handles edge cases consistently.
2. Wire `SemanticSimilarityExampleSelector` to your `FewShotChatMessagePromptTemplate` and select examples by closest match to the input.
3. Convert the JSON output to a Pydantic model with `with_structured_output(...)` — preview of Lab 2.4.
4. Build a 10-input eval set, run the inline few-shot prompt vs the `FewShotChatMessagePromptTemplate` version, and compare consistency.
